# Hotspots

The `concentration` findings describe one column at a time: a
dominant level, a near-constant. A combination of columns can
concentrate where neither column does — one `city` × `channel`
cell holding far more rows than its two marginals predict, in a
table where both marginals look unremarkable. The `hotspots`
detector reports those cells as `concentration` findings of type
`hotspot`. The detector is in the default set; the example
below names it in `detectors=[...]` to isolate the pass.


## The independence baseline

The scan runs over the shared encoded matrix — nominal columns
level-coded, numeric columns in the encoder's quantile bins —
for every pair of eligible features, and for triples at the deep
budget. For each cell it compares the observed row count with
the count expected under independence: the product of the cell's
marginal shares times the number of scanned rows. A cell is
reported when its standardized residual clears a Bonferroni
threshold over all cells examined at family α = 0.01, and it
also clears absolute bars on expected count, ratio, observed
count, and share of scanned rows.

Default eligibility requires between 2 and 20 non-missing bins or levels.
Most continuous columns use 32 bins and therefore do not enter this scan
without fewer distinct bin boundaries. Pairs may contain at most 1,024
cells, triples at most 4,096; deep examines up to 400 eligible triples.
When more triples are eligible than that cap allows, the examined
triples are chosen by a pre-score — the largest of the triple's
three pair scores from the pair scan — not by column order.

Counting examined cells matters. A wide pair space contains
thousands of cells; at an unadjusted bar, chance alone fills the
report with the luckiest of them. The Bonferroni denominator is
the number of cells actually counted, and the evidence states it.


In [1]:
import random

import polars as pl

import onceover


def frame(planted: bool, seed: int) -> pl.DataFrame:
    rng = random.Random(seed)
    if planted:
        rows = [("a1", "b1")] * 1000
        rest_a = ["a2", "a3", "a4", "a5"]
        rest_b = ["b2", "b3", "b4", "b5"]
        rows += [(rng.choice(rest_a), rng.choice(rest_b)) for _ in range(9000)]
        rng.shuffle(rows)
        return pl.DataFrame({"city": [r[0] for r in rows],
                             "channel": [r[1] for r in rows]})
    a = ["a1"] * 1000 + ["a2", "a3", "a4", "a5"] * 2250
    b = ["b1"] * 1000 + ["b2", "b3", "b4", "b5"] * 2250
    rng.shuffle(a)
    rng.shuffle(b)
    return pl.DataFrame({"city": a, "channel": b})


for label, planted, seed in (("planted cell", True, 11),
                             ("same marginals, independent", False, 12)):
    report = onceover.profile(frame(planted, seed), detectors=["hotspots"])
    hits = [f for f in report.findings if f.get("type") == "hotspot"]
    print(f"{label}: {len(hits)} hotspot findings")
    for f in hits:
        print("  ", f["claim"])
        print("   evidence:", f["evidence"][0]["metrics"])


planted cell: 1 hotspot findings
   cell (city = 'a1', channel = 'b1') holds 1000 of 10000 scanned rows (10.0%) against 100.0 expected under independence of city, channel — 10.0x the baseline (z = 90.0, bar 3.5 over 25 cells examined)
   evidence: {'cell': {'channel': 'b1', 'city': 'a1'}, 'cells_examined': 25, 'columns': ['city', 'channel'], 'expected_rows': 100.0, 'method': "observed cell count vs the independence baseline from the combination's margins over complete rows of the encoded sample; a cell reports only past absolute bars (expected >= 10, ratio >= 2, observed >= 20, share > 0.5%) and a Bonferroni z bar over all examined cells at family alpha 0.01", 'observed_rows': 1000, 'ratio': 10.0, 'rows_scanned': 10000, 'share': 0.1, 'z': 90.0, 'z_bar': 3.540083795298288}
same marginals, independent: 0 hotspot findings


## Reading the evidence

`cell` names the level (or bin) of each column in the
combination. `observed_rows` and `expected_rows` are counts over
the encoded row sample — the scan works on the sample the
encoder produced, and `rows_scanned` states its size.
`ratio` is observed over expected. `cells_examined` is the
Bonferroni denominator: every cell of every eligible combination
counted in the scan, reported so the bar can be re-derived.

If a deadline stops the scan, the pass's coverage entry reads
`capped` and states how many combinations and cells were examined
before the stop; the Bonferroni bar is computed over the examined
cells only, and a stop during the pair scan skips the triple phase
entirely. Findings from a stopped scan carry a `cells_evaluated`
metric giving the denominator the bar came from.


## Weighted concentration

Rows are not the only thing that can concentrate in a cell. A
cell can sit exactly at its row-count expectation while its
rows carry an outsized share of a quantity — rush-hour rows
are a small share of a bike-share table's rows and a much
larger share of its rentals. Row counting cannot see that, so
the pass also judges cells under declared weights: the
`weights` hint names numeric columns whose values weight the
rows, and several may be declared.

Under a weight, a cell is reported when the share of the
weight's total its rows carry is at least twice the cell's
share of rows, clears absolute bars on the cell's row count
and on the weight share, and clears a Bonferroni z bar over
the examined cells × weights at family α = 0.01. The z is
computed on the weight's effective row count, (Σw)² / Σw²,
because a total dominated by a few heavy rows carries less
information than the row count claims. The claim states both
shares and names the weight. The evidence adds
`expected_weight_share` — what the cell would hold if the
weight followed the features' own weight margins — and its
ratio `margin_ratio`, so a reader can separate the stated
concentration from what the margins already predict.

The weighted scan examines pairs over a wider eligibility
than the count scan — up to the encoder's 32 bins, so an
hour-of-day column's cells are in reach — and never judges a
weight over a combination containing the weight column
itself, whose bins would concentrate it by construction. A
declared weight that cannot be applied (not a column of the
table, not numeric, negative values, a zero total, or a
constant column, whose shares restate row shares) is not
applied, and the pass's coverage entry states each case.


In [2]:
def weighted_frame(seed: int = 21, n: int = 4000) -> pl.DataFrame:
    rng = random.Random(seed)
    cities = ["a1", "a2", "a3", "a4"]
    chans = ["b1", "b2", "b3", "b4"]
    rows = [(rng.choice(cities), rng.choice(chans)) for _ in range(n)]
    return pl.DataFrame({
        "city": [r[0] for r in rows],
        "channel": [r[1] for r in rows],
        "payout": [900.0 if r == ("a1", "b1") else 10.0 for r in rows],
    })


report = onceover.profile(weighted_frame(), detectors=["hotspots"],
                          hints={"weights": ["payout"]})
hits = [f for f in report.findings if f.get("type") == "hotspot"
        and "weight" in f["evidence"][0]["metrics"]]
print(f"{len(hits)} weighted hotspot findings (row counts are uniform)")
for f in hits:
    print("  ", f["claim"])


1 weighted hotspot findings (row counts are uniform)
   cell (city = 'a1', channel = 'b1') concentrates 86.4% of payout on 6.60% of scanned rows (264 of 4000) — 13.1x its row share (z = 60.4, bar 3.4 over 16 cell-weight tests examined)


## What the detector does not report

- **Single-column concentration.** A dominant level in one
  column is a `value_concentration` finding. Hotspots require a
  combination whose joint count exceeds what the marginals
  already explain — the independent frame in the example, with
  the same marginals as the planted frame, produces no finding.
- **Triples at the standard budget.** Three-column cells are
  scanned only at the deep budget; at standard the scan is pairs.
- **Combinations over the cell caps.** A pair or triple whose
  level combinations exceed the scan's cell caps is skipped
  rather than sampled down to fit.

- **Single-column weighted concentration.** One level of one
  column holding a large share of a weight is the weighted
  analog of `value_concentration`; the weighted scan reports
  combinations, so a weight's marginals are not stated.
